# Saber 11 (ICFES): resultados de las pruebas por estudiante o por municipio

El examen Saber 11 lo presentan los estudiantes al terminar la educación media. La base trae, para cada persona evaluada, los puntajes por área y el puntaje global, junto con datos del colegio (naturaleza, jornada, zona) y del hogar (estrato, educación de los padres, bienes). Es una de las bases más usadas para modelos de **clasificación** y **regresión** en educación.

|  |  |
|---|---|
| **Entidad** | Instituto Colombiano para la Evaluación de la Educación (ICFES) |
| **Conjunto** | Resultados únicos Saber 11 · [`kgxf-xxbe`](https://www.datos.gov.co/d/kgxf-xxbe) |
| **Una fila es** | una persona evaluada en un periodo (o, con `NIVEL = "municipio"`, un municipio en un periodo) |
| **Cobertura** | todo el país, por municipio de la sede del colegio |
| **Periodo** | 2010 a 2022; el puntaje global existe desde el segundo semestre de 2014 |
| **Tamaño total** | 7.109.704 filas y 51 columnas (consultado el 31/10/2026) |
| **Actualización** | la última carga es del 23/08/2023; no incluye 2023 en adelante |
| **Requisitos** | ninguno (token de datos.gov.co opcional) |
| **Tiempo de descarga** | 20 mil filas: menos de 1 minuto · un periodo completo del país (≈ 1 millón): 10 a 20 minutos |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base

Con `NIVEL = "estudiante"` cada fila es una persona evaluada. Con `NIVEL = "municipio"` el servidor calcula el número de estudiantes y el promedio de cada puntaje por municipio y periodo, y solo se descargan esos resúmenes.

| Columna | Qué es | Tipo |
|---|---|---|
| `periodo` | Año y aplicación: `20224` = 2022 calendario A; terminado en 1 = primer semestre (calendario B) | texto |
| `cole_cod_mcpio_ubicacion` → `cod_mpio` | Código DANE del municipio de la sede (5 dígitos) | texto |
| `cole_area_ubicacion` | Zona de la sede: URBANO o RURAL | categórica |
| `cole_naturaleza` | OFICIAL o NO OFICIAL | categórica |
| `cole_jornada` | MAÑANA, TARDE, COMPLETA, ÚNICA, NOCHE, SABATINA | categórica |
| `cole_calendario`, `cole_bilingue`, `cole_genero` | Calendario A/B, si es bilingüe, si es mixto o de un solo género | categórica |
| `estu_genero` | Género de la persona evaluada (F/M) | categórica |
| `fami_estratovivienda` | Estrato de la vivienda (Estrato 1 a 6, Sin Estrato) | categórica ordinal |
| `fami_educacionmadre`, `fami_educacionpadre` | Nivel educativo más alto de la madre y del padre | categórica ordinal |
| `fami_personashogar`, `fami_cuartoshogar` | Personas con las que vive y cuartos del hogar (rangos) | categórica ordinal |
| `fami_tieneinternet`, `fami_tienecomputador`, `fami_tieneautomovil`, `fami_tienelavadora` | Bienes del hogar (Si/No) | binaria |
| `punt_lectura_critica`, `punt_matematicas`, `punt_sociales_ciudadanas`, `punt_c_naturales`, `punt_ingles` | Puntaje por área (0 a 100) | número |
| `punt_global` | Puntaje global (0 a 500) | número |
| `anio` | Año, calculado a partir de `periodo` | número |

La base trae 51 columnas; la lista completa está en la página del conjunto. Para agregar una, escriba su nombre en `COLUMNAS`. Con `NIVEL = "municipio"` las columnas son `cod_mpio`, `cole_mcpio_ubicacion`, `cole_depto_ubicacion`, `periodo`, `estudiantes` y `prom_<área>` (promedio de cada puntaje).

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Clasificación | ¿El estudiante queda en el 25 % superior del puntaje global? | colegio (`cole_*`), hogar (`fami_*`), género |
| Regresión | `punt_global` o el puntaje de un área | colegio y hogar |
| Regresión (nivel municipio) | `prom_global` del municipio | unirlo con educación, agricultura o clima por `cod_mpio` |
| Agrupamiento | perfiles de municipios según sus promedios por área | los cinco `prom_*` escalados |

**Cuidados antes de modelar**

- **Fuga de información:** el puntaje global se calcula con los puntajes por área. Si predice `punt_global`, no use los `punt_*` como predictoras.
- Casi todas las predictoras son **categóricas**: necesitan codificación (one-hot u ordinal) o un modelo que las acepte, como `HistGradientBoosting`.
- Los periodos terminados en 1 (calendario B) tienen pocos estudiantes y un perfil distinto: no los mezcle sin una variable que los distinga.
- Para medir si el modelo sirve en otro territorio, valide **por municipio o departamento** (`GroupKFold`), no solo al azar.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

Periodos que tienen puntaje global: `20142`, `20151`, `20152`, `20161`, `20162`, `20171`, `20172`, `20181`, `20191`, `20194`, `20201`, `20211`, `20221` y `20224`. Los de mayor tamaño (calendario A, ≈ 550 mil a 1,1 millones de estudiantes) son `20152`, `20162`, `20172`, `20194` y `20224`.

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
NIVEL        = "estudiante"  # "estudiante": una fila por persona evaluada
                             # "municipio":  promedios por municipio y periodo (los calcula el servidor)
PERIODOS     = ["20224"]     # lista de periodos, p. ej. ["20194", "20224"]. None = todos los que tienen puntaje global
DEPARTAMENTO = "Meta"        # departamento de la sede del colegio, con o sin tildes. None = todo el país
MAX_FILAS    = 20_000        # tope de filas con NIVEL "estudiante". None = sin tope (un periodo del país ≈ 1 millón)

COLUMNAS = [                 # columnas para NIVEL "estudiante": quite o agregue nombres de la tabla de variables
    "periodo", "cole_cod_mcpio_ubicacion", "cole_mcpio_ubicacion", "cole_depto_ubicacion",
    "cole_area_ubicacion", "cole_naturaleza", "cole_jornada", "cole_calendario", "cole_bilingue", "cole_genero",
    "estu_genero", "fami_estratovivienda", "fami_educacionmadre", "fami_educacionpadre",
    "fami_personashogar", "fami_cuartoshogar", "fami_tieneinternet", "fami_tienecomputador",
    "fami_tieneautomovil", "fami_tienelavadora",
    "punt_lectura_critica", "punt_matematicas", "punt_sociales_ciudadanas", "punt_c_naturales",
    "punt_ingles", "punt_global",
]

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# Códigos DANE de los departamentos (para filtrar por código, que no depende de tildes)
COD_DPTO = {"ANTIOQUIA": "05", "ATLANTICO": "08", "BOGOTA": "11", "BOLIVAR": "13", "BOYACA": "15",
            "CALDAS": "17", "CAQUETA": "18", "CAUCA": "19", "CESAR": "20", "CORDOBA": "23",
            "CUNDINAMARCA": "25", "CHOCO": "27", "HUILA": "41", "LA GUAJIRA": "44", "MAGDALENA": "47",
            "META": "50", "NARINO": "52", "NORTE DE SANTANDER": "54", "QUINDIO": "63", "RISARALDA": "66",
            "SANTANDER": "68", "SUCRE": "70", "TOLIMA": "73", "VALLE DEL CAUCA": "76", "ARAUCA": "81",
            "CASANARE": "85", "PUTUMAYO": "86", "SAN ANDRES": "88", "AMAZONAS": "91", "GUAINIA": "94",
            "GUAVIARE": "95", "VAUPES": "97", "VICHADA": "99"}

def cod_dpto(nombre):
    """'Nariño' -> '52'."""
    c = canon(nombre)
    if c not in COD_DPTO:
        raise ValueError(f"No reconozco el departamento '{nombre}'. Use uno de: {sorted(COD_DPTO)}")
    return COD_DPTO[c]

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. ¿Cuántas filas cumplen el filtro?

Antes de descargar, se le pregunta al servidor cuántos estudiantes cumplen los parámetros.

In [ ]:
SABER = "kgxf-xxbe"
filtros = ["punt_global IS NOT NULL"]
if PERIODOS:
    filtros.append(f"periodo IN {lista_soql(PERIODOS)}")
if DEPARTAMENTO:
    filtros.append(f"cole_cod_depto_ubicacion = '{cod_dpto(DEPARTAMENTO)}'")
WHERE = " AND ".join(filtros)

n = contar(SABER, WHERE)
print("Filtro:", WHERE)
print(f"Estudiantes que cumplen el filtro: {entero(n)}")
if NIVEL == "estudiante" and MAX_FILAS and n > MAX_FILAS:
    print(f"Se descargarán solo {entero(MAX_FILAS)} (MAX_FILAS). No es una muestra al azar: "
          "para un análisis serio ponga MAX_FILAS = None o acote el filtro.")

## 5. Descarga

In [ ]:
PUNTAJES = ["punt_lectura_critica", "punt_matematicas", "punt_sociales_ciudadanas",
            "punt_c_naturales", "punt_ingles", "punt_global"]

if NIVEL == "estudiante":
    saber = socrata(SABER, select=", ".join(COLUMNAS), where=WHERE, max_rows=MAX_FILAS)
elif NIVEL == "municipio":
    promedios = ", ".join(f"avg({p}::number) as prom_{p[5:]}" for p in PUNTAJES)
    grupo = "cole_cod_mcpio_ubicacion, cole_mcpio_ubicacion, cole_depto_ubicacion, periodo"
    saber = socrata(SABER, select=f"{grupo}, count(*) as estudiantes, {promedios}", where=WHERE, group=grupo)
else:
    raise ValueError('NIVEL debe ser "estudiante" o "municipio"')
print(f"Descargadas {entero(len(saber))} filas")

## 6. Limpieza mínima

Se pasan los puntajes a número, el código del municipio a 5 dígitos y se agrega el año.

In [ ]:
if "cole_cod_mcpio_ubicacion" in saber:
    saber.insert(0, "cod_mpio", cod5(saber.pop("cole_cod_mcpio_ubicacion")))
saber["anio"] = saber["periodo"].astype(str).str[:4].astype(int)
saber = a_numero(saber, [c for c in saber.columns if c.startswith(("punt_", "prom_")) or c == "estudiantes"])

## 7. Revisión de la base

In [ ]:
ficha_rapida(saber)
saber.head()

*(Se completa al ejecutar.)*

In [ ]:
y = "punt_global" if NIVEL == "estudiante" else "prom_global"
corte = saber[y].quantile(0.75)

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(saber[y].dropna(), bins=50, color=AZUL, edgecolor="white", linewidth=0.4)
ax.axvline(corte, color=NARANJA, lw=2)
ax.text(corte, ax.get_ylim()[1] * 0.9, f"  25 % superior: {entero(corte)} o más", color=NARANJA)
ax.set_title(f"Distribución de {y} — {DEPARTAMENTO or 'país'}")
ax.set_xlabel("puntaje global (0 a 500)")
ax.set_ylabel("estudiantes" if NIVEL == "estudiante" else "municipios")
plt.tight_layout(); plt.show()

if NIVEL == "estudiante" and "fami_estratovivienda" in saber:
    display(saber.groupby("fami_estratovivienda")["punt_global"]
                 .agg(estudiantes="size", promedio="mean").round(1))

*(Se completa al ejecutar.)*

## 8. Guardar

In [ ]:
ruta = guardar(saber, etiqueta("saber11", NIVEL, DEPARTAMENTO or "pais"))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/saber11_estudiante_meta.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.
5. Para una variable objetivo binaria: `saber["alto"] = (saber["punt_global"] >= saber["punt_global"].quantile(0.75)).astype(int)`.